# 🎬 AI Studio — Kaggle GPU Backend
**Models supported:** LTX Video 2B, Wan 2.1 (1.3B / 14B), + any future ComfyUI model  
**GPU:** P100 (16GB) default — works for LTX 2B and Wan 1.3B  

> At the end of setup, a public URL will be printed. Open it to use the studio UI.

In [ ]:
# ============================================================
# CELL 1 — Configuration
# Toggle which models to download (saves time & disk space)
# ============================================================

DOWNLOAD_LTX_2B        = True   # ~9GB model + ~9GB T5 clip — fits P100
DOWNLOAD_WAN_1_3B      = True   # ~5GB  — fits P100 easily
DOWNLOAD_WAN_14B       = False  # ~28GB — needs A100/H100
DOWNLOAD_LTX_13B       = False  # ~26GB — needs A100/H100
DOWNLOAD_FIRERED       = True   # ~15GB — FireRed Image Edit 1.1 (Qwen2-VL). Needs 14GB+ VRAM.
DOWNLOAD_WAN_I2V       = False  # ~14GB — Wan 2.1 I2V 480P
DOWNLOAD_WAN22_T2I     = False  # ~20GB — Wan 2.2 T2I 14B (dual UNETLoader). Needs A100.
DOWNLOAD_WAN22_I2V     = False  # ~20GB — Wan 2.2 I2V 14B 10-step. Needs A100.

# HuggingFace token (needed for gated models — paste yours here or leave blank)
HF_TOKEN = ""

# Studio port (what the browser connects to)
STUDIO_PORT = 7860
COMFY_PORT  = 8188

# Working directory
import os
WORK_DIR   = "/kaggle/working"
COMFY_DIR  = f"{WORK_DIR}/ComfyUI"
STUDIO_DIR = f"{WORK_DIR}/ai-studio"

print("Configuration set.")
print(f"  LTX 2B:      {DOWNLOAD_LTX_2B}")
print(f"  Wan 1.3B:    {DOWNLOAD_WAN_1_3B}")
print(f"  Wan 14B:     {DOWNLOAD_WAN_14B}")
print(f"  Wan I2V:     {DOWNLOAD_WAN_I2V}")
print(f"  FireRed:     {DOWNLOAD_FIRERED}")
print(f"  Wan 2.2 T2I: {DOWNLOAD_WAN22_T2I}")
print(f"  Wan 2.2 I2V: {DOWNLOAD_WAN22_I2V}")

In [ ]:
# ============================================================
# CELL 2 — Install system dependencies
# ============================================================

import subprocess, sys

def run(cmd, **kw):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=False, **kw)
    if result.returncode != 0:
        print(f"⚠️  Command returned {result.returncode}")
    return result

# System packages
run("apt-get update -qq && apt-get install -y -qq ffmpeg libgl1-mesa-glx libglib2.0-0 wget curl")

# Python packages for the studio backend
run("pip install -q fastapi uvicorn[standard] python-multipart aiohttp aiofiles websockets Pillow huggingface_hub")

# FireRed dependencies
# NOTE: transformers MUST be pinned at 4.57.6 for QwenImageEditPlusPipeline compatibility
# NOTE: diffusers must be installed from git for QwenImageEditPlusPipeline support
if DOWNLOAD_FIRERED:
    print("\n📦 Installing FireRed dependencies…")
    run("pip install -q transformers==4.57.6 accelerate peft sentencepiece")
    run("pip install -q git+https://github.com/huggingface/diffusers.git")
    print("✅ FireRed dependencies installed.")

print("\n✅ System dependencies installed.")

In [ ]:
# ============================================================
# CELL 3 — Install ComfyUI
# ============================================================

import os

if not os.path.exists(COMFY_DIR):
    run(f"git clone --depth=1 https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}")
else:
    print("ComfyUI already cloned, pulling latest…")
    run(f"git -C {COMFY_DIR} pull --ff-only")

run(f"pip install -q -r {COMFY_DIR}/requirements.txt")

# Create model directories
for d in [
    f"{COMFY_DIR}/models/checkpoints",
    f"{COMFY_DIR}/models/clip",
    f"{COMFY_DIR}/models/vae",
    f"{COMFY_DIR}/models/loras",
    f"{COMFY_DIR}/models/wan",
    f"{COMFY_DIR}/input",
    f"{COMFY_DIR}/output",
]:
    os.makedirs(d, exist_ok=True)

print("\n✅ ComfyUI ready.")

In [ ]:
# ============================================================
# CELL 4 — Install Custom Nodes
# ============================================================

CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

custom_nodes = [
    # Always install
    ("ComfyUI-Manager",         "https://github.com/ltdrdata/ComfyUI-Manager"),
    ("ComfyUI-VideoHelperSuite","https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite"),
]

if DOWNLOAD_LTX_2B or DOWNLOAD_LTX_13B:
    custom_nodes.append(("ComfyUI-LTXVideo", "https://github.com/Lightricks/ComfyUI-LTXVideo"))

if DOWNLOAD_WAN_1_3B or DOWNLOAD_WAN_14B:
    custom_nodes.append(("ComfyUI-WanVideoWrapper", "https://github.com/kijai/ComfyUI-WanVideoWrapper"))

for name, url in custom_nodes:
    dest = f"{CUSTOM_NODES_DIR}/{name}"
    if not os.path.exists(dest):
        print(f"Installing {name}…")
        run(f"git clone --depth=1 {url} {dest}")
        req = f"{dest}/requirements.txt"
        if os.path.exists(req):
            run(f"pip install -q -r {req}")
    else:
        print(f"  {name} already installed")

print("\n✅ Custom nodes installed.")

In [ ]:
# ============================================================
# CELL 5 — Download Models
# ============================================================

from huggingface_hub import hf_hub_download, snapshot_download
import os

hf_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}

def download_file(repo_id, filename, dest_path, repo_type="model"):
    if os.path.exists(dest_path):
        print(f"  ✓ Already exists: {os.path.basename(dest_path)}")
        return
    print(f"  ↓ Downloading {filename} from {repo_id}…")
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    hf_hub_download(repo_id=repo_id, filename=filename,
                    local_dir=os.path.dirname(dest_path),
                    repo_type=repo_type, **hf_kwargs)
    print(f"  ✅ Done: {os.path.basename(dest_path)}")

def download_snapshot(repo_id, dest_dir, repo_type="model"):
    if os.path.exists(dest_dir) and len(os.listdir(dest_dir)) > 2:
        print(f"  ✓ Already exists: {dest_dir}")
        return
    print(f"  ↓ Downloading snapshot {repo_id}…")
    os.makedirs(dest_dir, exist_ok=True)
    snapshot_download(repo_id=repo_id, local_dir=dest_dir,
                      repo_type=repo_type, **hf_kwargs)
    print(f"  ✅ Done: {repo_id}")

# Ensure diffusion_models directory exists (for UNETLoader-based models)
os.makedirs(f"{COMFY_DIR}/models/diffusion_models", exist_ok=True)

# --- LTX Video 2B ---
if DOWNLOAD_LTX_2B:
    print("\n📦 LTX Video 2B:")
    download_file("Lightricks/LTX-Video", "ltx-video-2b-v0.9.5.safetensors",
                  f"{COMFY_DIR}/models/checkpoints/ltx-video-2b-v0.9.5.safetensors")
    download_file("comfyanonymous/flux_text_encoders", "t5xxl_fp16.safetensors",
                  f"{COMFY_DIR}/models/clip/t5xxl_fp16.safetensors")

# --- LTX Video 13B ---
if DOWNLOAD_LTX_13B:
    print("\n📦 LTX Video 13B:")
    download_file("Lightricks/LTX-Video", "ltx-video-13b-v0.9.5.safetensors",
                  f"{COMFY_DIR}/models/checkpoints/ltx-video-13b-v0.9.5.safetensors")

# --- Wan 2.1 T2V 1.3B ---
if DOWNLOAD_WAN_1_3B:
    print("\n📦 Wan 2.1 T2V 1.3B:")
    download_snapshot("Wan-AI/Wan2.1-T2V-1.3B", f"{COMFY_DIR}/models/wan/Wan2.1-T2V-1.3B")

# --- Wan 2.1 T2V 14B ---
if DOWNLOAD_WAN_14B:
    print("\n📦 Wan 2.1 T2V 14B:")
    download_snapshot("Wan-AI/Wan2.1-T2V-14B", f"{COMFY_DIR}/models/wan/Wan2.1-T2V-14B")

# --- Wan 2.1 I2V 480P ---
if DOWNLOAD_WAN_I2V:
    print("\n📦 Wan 2.1 I2V 480P:")
    download_snapshot("Wan-AI/Wan2.1-I2V-14B-480P", f"{COMFY_DIR}/models/wan/Wan2.1-I2V-14B-480P")

# --- Wan 2.2 T2I 14B ---
# Shared files (text encoder + VAE) used by both Wan 2.2 T2I and I2V
def download_wan22_shared():
    download_file("Comfy-Org/Wan_2.1_ComfyUI_repackaged",
                  "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
                  f"{COMFY_DIR}/models/clip/umt5_xxl_fp8_e4m3fn_scaled.safetensors")
    download_file("Comfy-Org/Wan_2.1_ComfyUI_repackaged",
                  "split_files/vae/wan_2.1_vae.safetensors",
                  f"{COMFY_DIR}/models/vae/wan_2.1_vae.safetensors")

if DOWNLOAD_WAN22_T2I:
    print("\n📦 Wan 2.2 T2I 14B:")
    download_wan22_shared()
    # Wan 2.2 T2V models from kijai's fp8 repo
    download_file("kijai/WanVideo-fp8",
                  "Wan2_2-T2V-A14B_HIGH_fp8_e4m3fn_scaled_KJ.safetensors",
                  f"{COMFY_DIR}/models/diffusion_models/Wan2_2-T2V-A14B_HIGH_fp8_e4m3fn_scaled_KJ.safetensors")
    download_file("kijai/WanVideo-fp8",
                  "Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
                  f"{COMFY_DIR}/models/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors")

# --- Wan 2.2 I2V 14B (10-step) ---
if DOWNLOAD_WAN22_I2V:
    print("\n📦 Wan 2.2 I2V 14B (10-step):")
    download_wan22_shared()
    # Wan 2.2 I2V 10-step models from StefanFalkok's dataset
    download_file("StefanFalkok/Wan_2.2_I2V_10steps",
                  "Wan_2.2_I2V_HighNoise_10steps_fp8.safetensors",
                  f"{COMFY_DIR}/models/diffusion_models/Wan_2.2_I2V_HighNoise_10steps_fp8.safetensors",
                  repo_type="dataset")
    download_file("StefanFalkok/Wan_2.2_I2V_10steps",
                  "Wan_2.2_I2V_LowNoise_10steps_fp8.safetensors",
                  f"{COMFY_DIR}/models/diffusion_models/Wan_2.2_I2V_LowNoise_10steps_fp8.safetensors",
                  repo_type="dataset")

# --- FireRed Image Edit 1.1 ---
if DOWNLOAD_FIRERED:
    print("\n📦 FireRed Image Edit 1.1 (pre-caching weights)…")
    print("   This downloads ~15GB. FireRed requires 14GB+ VRAM (A100 recommended).")
    download_snapshot("FireRedTeam/FireRed-Image-Edit-1.1",
                      f"{WORK_DIR}/hf_cache/FireRed-Image-Edit-1.1")
    download_snapshot("prithivMLmods/Qwen-Image-Edit-Rapid-AIO-V19",
                      f"{WORK_DIR}/hf_cache/Qwen-Image-Edit-Rapid-AIO-V19")
    os.environ["HUGGINGFACE_HUB_CACHE"] = f"{WORK_DIR}/hf_cache"
    os.environ["FIRERED_MODEL_ID"] = f"{WORK_DIR}/hf_cache/FireRed-Image-Edit-1.1"
    os.environ["FIRERED_TRANSFORMER_ID"] = f"{WORK_DIR}/hf_cache/Qwen-Image-Edit-Rapid-AIO-V19"
    print("  ✅ FireRed weights cached.")

print("\n✅ All model downloads complete.")

In [ ]:
# ============================================================
# CELL 6 — Clone AI Studio backend
# ============================================================

import os, json

os.makedirs(f"{STUDIO_DIR}/backend/workflows", exist_ok=True)
os.makedirs(f"{STUDIO_DIR}/frontend", exist_ok=True)

GITHUB_RAW = ""  # e.g. "https://raw.githubusercontent.com/you/ai-studio/main"

if GITHUB_RAW:
    for f in ["backend/server.py", "backend/comfy_client.py",
              "backend/workflow_builder.py", "backend/requirements.txt",
              "frontend/index.html", "models_config.json"]:
        run(f"wget -q -O {STUDIO_DIR}/{f} {GITHUB_RAW}/{f}")
    print("✅ Studio files downloaded from GitHub.")
else:
    print("ℹ️  Set GITHUB_RAW above, or upload studio files as a Kaggle dataset.")

# Write models_config.json inline (always)
models_config = {
  "models": {
    "firered-1.1": {
      "id": "firered-1.1",
      "name": "FireRed Edit 1.1",
      "category": "image",
      "mode": "i2i",
      "label": "Image Edit",
      "description": "FireRedTeam FireRed 1.1 — Qwen2-VL image editing. Give it an image + instruction and it edits it.",
      "backend": "diffusers",
      "hf_model_id": "FireRedTeam/FireRed-Image-Edit-1.1",
      "hf_transformer_id": "prithivMLmods/Qwen-Image-Edit-Rapid-AIO-V19",
      "param_nodes": {},
      "defaults": {"steps": 4, "cfg": 1.0},
      "aspect_ratios": {},
      "supports_first_frame": False,
      "supports_last_frame": False,
      "vram_gb": 14,
      "custom_nodes": [],
      "download_toggle": "DOWNLOAD_FIRERED",
      "prompt_label": "Edit instruction (e.g. 'make the sky purple and add rain')"
    },
    "wan-2.2-t2i": {
      "id": "wan-2.2-t2i",
      "name": "Wan 2.2 T2I 14B",
      "category": "image",
      "mode": "t2i",
      "label": "Text → Image",
      "description": "Alibaba Wan 2.2 14B — high quality text-to-image, dual-pass sampling",
      "workflow_file": "wan22_t2i.json",
      "param_nodes": {
        "positive_prompt": {"node_id": "3", "field": "text"},
        "negative_prompt": {"node_id": "4", "field": "text"},
        "width":  {"node_id": "5", "field": "width"},
        "height": {"node_id": "5", "field": "height"},
        "steps":  {"node_id": "35", "field": "steps"},
        "cfg":    {"node_id": "35", "field": "cfg"},
        "seed":   {"node_id": "35", "field": "noise_seed"}
      },
      "defaults": {"width": 1024, "height": 1024, "steps": 10, "cfg": 3.5},
      "aspect_ratios": {
        "1:1":  {"w": 1024, "h": 1024},
        "16:9": {"w": 1280, "h": 720},
        "9:16": {"w": 720,  "h": 1280},
        "4:3":  {"w": 1152, "h": 864},
        "3:4":  {"w": 864,  "h": 1152}
      },
      "supports_first_frame": False,
      "supports_last_frame": False,
      "vram_gb": 20,
      "custom_nodes": [],
      "download_toggle": "DOWNLOAD_WAN22_T2I"
    },
    "wan-2.2-i2v": {
      "id": "wan-2.2-i2v",
      "name": "Wan 2.2 I2V 14B",
      "category": "video",
      "mode": "i2v",
      "label": "Image → Video",
      "description": "Alibaba Wan 2.2 14B — animate images, 10-step dual-pass sampling",
      "workflow_file": "wan22_i2v.json",
      "param_nodes": {
        "positive_prompt": {"node_id": "6", "field": "text"},
        "negative_prompt": {"node_id": "7", "field": "text"},
        "width":      {"node_id": "50", "field": "width"},
        "height":     {"node_id": "50", "field": "height"},
        "frames":     {"node_id": "50", "field": "length"},
        "seed":       {"node_id": "185", "field": "noise_seed"},
        "init_image": {"node_id": "214", "field": "image"}
      },
      "defaults": {"width": 832, "height": 480, "frames": 81, "steps": 10, "cfg": 2.0, "fps": 16},
      "aspect_ratios": {
        "1:1":  {"w": 832,  "h": 832},
        "16:9": {"w": 1024, "h": 576},
        "9:16": {"w": 576,  "h": 1024},
        "4:3":  {"w": 832,  "h": 480},
        "3:4":  {"w": 544,  "h": 720}
      },
      "supports_first_frame": False,
      "supports_last_frame": False,
      "fps": 16,
      "max_frames": 81,
      "vram_gb": 20,
      "custom_nodes": ["ComfyUI-VideoHelperSuite"],
      "download_toggle": "DOWNLOAD_WAN22_I2V"
    },
    "ltx-video-2b-t2v": {
      "id": "ltx-video-2b-t2v",
      "name": "LTX Video 2B",
      "category": "video",
      "mode": "t2v",
      "label": "Text → Video",
      "description": "Lightricks LTX-Video 2B — fast, high quality. Supports pinned first/last frames.",
      "workflow_file": "ltx_t2v.json",
      "param_nodes": {
        "positive_prompt": {"node_id": "6", "field": "text"},
        "negative_prompt": {"node_id": "7", "field": "text"},
        "width": {"node_id": "5", "field": "width"},
        "height": {"node_id": "5", "field": "height"},
        "frames": {"node_id": "5", "field": "video_frames"},
        "steps": {"node_id": "3", "field": "steps"},
        "cfg": {"node_id": "3", "field": "cfg"},
        "seed": {"node_id": "3", "field": "noise_seed"},
        "model_name": {"node_id": "1", "field": "ckpt_name"},
        "clip_name": {"node_id": "2", "field": "clip_name"}
      },
      "lora_insert_after_node": "1",
      "defaults": {"width": 768, "height": 512, "frames": 97, "steps": 30, "cfg": 3.0, "fps": 24},
      "aspect_ratios": {
        "1:1": {"w": 512, "h": 512},
        "16:9": {"w": 768, "h": 512},
        "9:16": {"w": 512, "h": 768},
        "4:3": {"w": 704, "h": 528},
        "3:4": {"w": 528, "h": 704},
        "21:9": {"w": 1024, "h": 432}
      },
      "supports_first_frame": True,
      "supports_last_frame": True,
      "first_frame_node": "LTXVAddGuide",
      "first_frame_positive_node_id": "6",
      "last_frame_positive_node_id": "6",
      "fps": 24,
      "max_frames": 121,
      "vram_gb": 10,
      "custom_nodes": ["ComfyUI-LTXVideo", "ComfyUI-VideoHelperSuite"],
      "download_toggle": "DOWNLOAD_LTX_2B"
    },
    "ltx-video-2b-i2v": {
      "id": "ltx-video-2b-i2v",
      "name": "LTX Video 2B I2V",
      "category": "video",
      "mode": "i2v",
      "label": "Image → Video",
      "description": "Lightricks LTX-Video 2B — animate a still image into video",
      "workflow_file": "ltx_i2v.json",
      "param_nodes": {
        "positive_prompt": {"node_id": "6", "field": "text"},
        "negative_prompt": {"node_id": "7", "field": "text"},
        "width": {"node_id": "5", "field": "width"},
        "height": {"node_id": "5", "field": "height"},
        "frames": {"node_id": "5", "field": "video_frames"},
        "steps": {"node_id": "3", "field": "steps"},
        "cfg": {"node_id": "3", "field": "cfg"},
        "seed": {"node_id": "3", "field": "noise_seed"}
      },
      "lora_insert_after_node": "1",
      "defaults": {"width": 768, "height": 512, "frames": 97, "steps": 30, "cfg": 3.0, "fps": 24},
      "aspect_ratios": {
        "1:1": {"w": 512, "h": 512},
        "16:9": {"w": 768, "h": 512},
        "9:16": {"w": 512, "h": 768},
        "4:3": {"w": 704, "h": 528},
        "3:4": {"w": 528, "h": 704}
      },
      "supports_first_frame": False,
      "supports_last_frame": True,
      "fps": 24,
      "max_frames": 121,
      "vram_gb": 10,
      "custom_nodes": ["ComfyUI-LTXVideo", "ComfyUI-VideoHelperSuite"],
      "download_toggle": "DOWNLOAD_LTX_2B"
    },
    "wan-1.3b-t2v": {
      "id": "wan-1.3b-t2v",
      "name": "Wan 2.1 T2V 1.3B",
      "category": "video",
      "mode": "t2v",
      "label": "Text → Video",
      "description": "Alibaba Wan 2.1 — 1.3B, optimized for P100 (16GB VRAM)",
      "workflow_file": "wan_t2v.json",
      "param_nodes": {
        "positive_prompt": {"node_id": "6", "field": "text"},
        "negative_prompt": {"node_id": "7", "field": "text"},
        "width": {"node_id": "5", "field": "width"},
        "height": {"node_id": "5", "field": "height"},
        "frames": {"node_id": "5", "field": "video_frames"},
        "steps": {"node_id": "3", "field": "steps"},
        "cfg": {"node_id": "3", "field": "guidance_scale"},
        "seed": {"node_id": "3", "field": "seed"},
        "model_path": {"node_id": "1", "field": "model_path"}
      },
      "defaults": {"width": 832, "height": 480, "frames": 81, "steps": 30, "cfg": 6.0, "fps": 16},
      "aspect_ratios": {
        "1:1": {"w": 480, "h": 480},
        "16:9": {"w": 832, "h": 480},
        "9:16": {"w": 480, "h": 832},
        "4:3": {"w": 640, "h": 480},
        "3:4": {"w": 480, "h": 640}
      },
      "supports_first_frame": False,
      "supports_last_frame": False,
      "fps": 16,
      "max_frames": 121,
      "vram_gb": 8,
      "custom_nodes": ["ComfyUI-WanVideoWrapper", "ComfyUI-VideoHelperSuite"],
      "download_toggle": "DOWNLOAD_WAN_1_3B"
    },
    "wan-1.3b-i2v": {
      "id": "wan-1.3b-i2v",
      "name": "Wan 2.1 I2V 1.3B",
      "category": "video",
      "mode": "i2v",
      "label": "Image → Video",
      "description": "Alibaba Wan 2.1 — animate images, P100 compatible",
      "workflow_file": "wan_i2v.json",
      "param_nodes": {
        "positive_prompt": {"node_id": "6", "field": "text"},
        "negative_prompt": {"node_id": "7", "field": "text"},
        "width": {"node_id": "5", "field": "width"},
        "height": {"node_id": "5", "field": "height"},
        "frames": {"node_id": "5", "field": "video_frames"},
        "steps": {"node_id": "3", "field": "steps"},
        "cfg": {"node_id": "3", "field": "guidance_scale"},
        "seed": {"node_id": "3", "field": "seed"}
      },
      "defaults": {"width": 832, "height": 480, "frames": 81, "steps": 30, "cfg": 6.0, "fps": 16},
      "aspect_ratios": {
        "1:1": {"w": 480, "h": 480},
        "16:9": {"w": 832, "h": 480},
        "9:16": {"w": 480, "h": 832},
        "4:3": {"w": 640, "h": 480},
        "3:4": {"w": 480, "h": 640}
      },
      "supports_first_frame": False,
      "supports_last_frame": False,
      "fps": 16,
      "max_frames": 121,
      "vram_gb": 12,
      "custom_nodes": ["ComfyUI-WanVideoWrapper", "ComfyUI-VideoHelperSuite"],
      "download_toggle": "DOWNLOAD_WAN_I2V"
    }
  },
  "custom_nodes": {
    "ComfyUI-LTXVideo":         {"repo": "https://github.com/Lightricks/ComfyUI-LTXVideo"},
    "ComfyUI-WanVideoWrapper":  {"repo": "https://github.com/kijai/ComfyUI-WanVideoWrapper"},
    "ComfyUI-VideoHelperSuite": {"repo": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite"},
    "ComfyUI-Manager":          {"repo": "https://github.com/ltdrdata/ComfyUI-Manager"}
  }
}

with open(f"{STUDIO_DIR}/models_config.json", "w") as f:
    json.dump(models_config, f, indent=2)

print("✅ models_config.json written.")

In [ ]:
# ============================================================
# CELL 7 — Write studio server files (self-contained inline)
# This cell writes all the Python backend files directly.
# If you're pulling from GitHub, you can skip this cell.
# ============================================================

import os, shutil

BACKEND_DIR = f"{STUDIO_DIR}/backend"
os.makedirs(f"{BACKEND_DIR}/workflows", exist_ok=True)

# --- requirements.txt ---
with open(f"{BACKEND_DIR}/requirements.txt", "w") as f:
    f.write("fastapi>=0.111.0\nuvicorn[standard]>=0.29.0\npython-multipart>=0.0.9\naiohttp>=3.9.0\naiofiles>=23.2.1\nwebsockets>=12.0\nPillow>=10.0.0\n")

run(f"pip install -q -r {BACKEND_DIR}/requirements.txt")

# --- Copy backend Python files ---
# Priority order: Kaggle dataset input → GitHub (if GITHUB_RAW set) → warn
for src_name in ["comfy_client.py", "workflow_builder.py", "server.py", "firered.py"]:
    kaggle_input = f"/kaggle/input/ai-studio-backend/{src_name}"
    dest = f"{BACKEND_DIR}/{src_name}"
    if os.path.exists(kaggle_input):
        shutil.copy(kaggle_input, dest)
        print(f"  ✓ Copied {src_name} from Kaggle input")
    elif os.path.exists(dest):
        print(f"  ✓ {src_name} already present")
    else:
        print(f"  ⚠️  {src_name} not found — upload ~/ai-studio/backend/ as a Kaggle dataset")

# --- Copy workflow JSON files ---
WORKFLOWS_SRC = "/kaggle/input/ai-studio-backend/workflows"
if os.path.isdir(WORKFLOWS_SRC):
    for wf in os.listdir(WORKFLOWS_SRC):
        if wf.endswith(".json"):
            shutil.copy(f"{WORKFLOWS_SRC}/{wf}", f"{BACKEND_DIR}/workflows/{wf}")
    print(f"  ✓ Workflow files copied from Kaggle input")

# --- Copy frontend ---
FRONTEND_SRC = "/kaggle/input/ai-studio-backend/frontend"
if os.path.isdir(FRONTEND_SRC):
    os.makedirs(f"{STUDIO_DIR}/frontend", exist_ok=True)
    for f in os.listdir(FRONTEND_SRC):
        shutil.copy(f"{FRONTEND_SRC}/{f}", f"{STUDIO_DIR}/frontend/{f}")
    print(f"  ✓ Frontend files copied from Kaggle input")

# --- Default workflow templates (inline fallback) ---
import json

ltx_t2v = {
  "1": {"inputs": {"ckpt_name": "ltx-video-2b-v0.9.5.safetensors"}, "class_type": "LTXVLoader"},
  "2": {"inputs": {"clip_name": "t5xxl_fp16.safetensors", "type": "ltxv"}, "class_type": "CLIPLoader"},
  "6": {"inputs": {"text": "cinematic video", "clip": ["2", 0]}, "class_type": "CLIPTextEncode"},
  "7": {"inputs": {"text": "worst quality, blurry", "clip": ["2", 0]}, "class_type": "CLIPTextEncode"},
  "5": {"inputs": {"width": 768, "height": 512, "video_frames": 97, "batch_size": 1}, "class_type": "LTXVEmptyLatentVideo"},
  "3": {"inputs": {"model": ["1", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["5", 0],
         "noise_seed": 42, "steps": 30, "cfg": 3.0, "sampler_name": "euler", "scheduler": "ltv_uniform"},
       "class_type": "LTXVSampler"},
  "9": {"inputs": {"samples": ["3", 0], "vae": ["1", 2]}, "class_type": "VAEDecode"},
  "10": {"inputs": {"frame_rate": 24, "loop_count": 0, "filename_prefix": "ltx_video",
          "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": 19,
          "save_metadata": True, "pingpong": False, "save_output": True, "images": ["9", 0]},
        "class_type": "VHS_VideoCombine"}
}

wan_t2v = {
  "1": {"inputs": {"model_path": "models/wan/Wan2.1-T2V-1.3B"}, "class_type": "WanVideoModelLoader"},
  "6": {"inputs": {"text": "a beautiful landscape", "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
  "7": {"inputs": {"text": "worst quality", "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
  "5": {"inputs": {"width": 832, "height": 480, "video_frames": 81, "batch_size": 1}, "class_type": "WanVideoEmptyLatent"},
  "3": {"inputs": {"model": ["1", 0], "positive": ["6", 0], "negative": ["7", 0], "latents": ["5", 0],
         "steps": 30, "guidance_scale": 6.0, "seed": 42}, "class_type": "WanVideoSampler"},
  "9": {"inputs": {"samples": ["3", 0], "vae": ["1", 2]}, "class_type": "VAEDecode"},
  "10": {"inputs": {"frame_rate": 16, "loop_count": 0, "filename_prefix": "wan_video",
          "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": 19,
          "save_metadata": True, "pingpong": False, "save_output": True, "images": ["9", 0]},
        "class_type": "VHS_VideoCombine"}
}

wan22_t2i = {
  "49": {"inputs": {"unet_name": "Wan2_2-T2V-A14B_HIGH_fp8_e4m3fn_scaled_KJ.safetensors", "weight_dtype": "default"}, "class_type": "UNETLoader"},
  "50": {"inputs": {"unet_name": "Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors", "weight_dtype": "default"}, "class_type": "UNETLoader"},
  "22": {"inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}, "class_type": "CLIPLoader"},
  "8":  {"inputs": {"vae_name": "wan_2.1_vae.safetensors"}, "class_type": "VAELoader"},
  "3":  {"inputs": {"text": "a beautiful landscape", "clip": ["22", 0]}, "class_type": "CLIPTextEncode"},
  "4":  {"inputs": {"text": "low quality, blurry", "clip": ["22", 0]}, "class_type": "CLIPTextEncode"},
  "5":  {"inputs": {"width": 1024, "height": 1024, "video_length": 1, "batch_size": 1}, "class_type": "EmptyHunyuanLatentVideo"},
  "35": {"inputs": {"model": ["49", 0], "positive": ["3", 0], "negative": ["4", 0], "latent_image": ["5", 0],
                    "add_noise": "enable", "noise_seed": 42, "control_after_generate": "fixed",
                    "steps": 10, "cfg": 3.5, "sampler_name": "euler", "scheduler": "simple",
                    "start_at_step": 0, "end_at_step": 3, "return_with_leftover_noise": "disable"},
         "class_type": "KSamplerAdvanced"},
  "36": {"inputs": {"model": ["50", 0], "positive": ["3", 0], "negative": ["4", 0], "latent_image": ["35", 0],
                    "add_noise": "enable", "noise_seed": 42, "control_after_generate": "fixed",
                    "steps": 10, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple",
                    "start_at_step": 3, "end_at_step": 10, "return_with_leftover_noise": "disable"},
         "class_type": "KSamplerAdvanced"},
  "9":  {"inputs": {"samples": ["36", 0], "vae": ["8", 0]}, "class_type": "VAEDecode"},
  "10": {"inputs": {"images": ["9", 0], "filename_prefix": "wan22_t2i"}, "class_type": "SaveImage"}
}

wan22_i2v = {
  "231": {"inputs": {"unet_name": "Wan_2.2_I2V_HighNoise_10steps_fp8.safetensors", "weight_dtype": "default"}, "class_type": "UNETLoader"},
  "230": {"inputs": {"unet_name": "Wan_2.2_I2V_LowNoise_10steps_fp8.safetensors", "weight_dtype": "default"}, "class_type": "UNETLoader"},
  "54":  {"inputs": {"model": ["231", 0], "shift": 8}, "class_type": "ModelSamplingSD3"},
  "187": {"inputs": {"model": ["230", 0], "shift": 8}, "class_type": "ModelSamplingSD3"},
  "38":  {"inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}, "class_type": "CLIPLoader"},
  "39":  {"inputs": {"vae_name": "wan_2.1_vae.safetensors"}, "class_type": "VAELoader"},
  "6":   {"inputs": {"text": "a person walking", "clip": ["38", 0]}, "class_type": "CLIPTextEncode"},
  "7":   {"inputs": {"text": "low quality, blurry, distorted anatomy", "clip": ["38", 0]}, "class_type": "CLIPTextEncode"},
  "214": {"inputs": {"image": "example.png", "upload": "image"}, "class_type": "LoadImage"},
  "50":  {"inputs": {"positive": ["6", 0], "negative": ["7", 0], "vae": ["39", 0], "start_image": ["214", 0],
                     "width": 832, "height": 480, "length": 81, "batch_size": 1},
          "class_type": "WanImageToVideo"},
  "185": {"inputs": {"model": ["54", 0], "positive": ["50", 0], "negative": ["50", 1], "latent_image": ["50", 2],
                     "add_noise": "enable", "noise_seed": 42, "control_after_generate": "fixed",
                     "steps": 10, "cfg": 2.0, "sampler_name": "uni_pc", "scheduler": "simple",
                     "start_at_step": 0, "end_at_step": 5, "return_with_leftover_noise": "disable"},
          "class_type": "KSamplerAdvanced"},
  "191": {"inputs": {"model": ["187", 0], "positive": ["50", 0], "negative": ["50", 1], "latent_image": ["185", 0],
                     "add_noise": "disable", "noise_seed": 0, "control_after_generate": "fixed",
                     "steps": 10, "cfg": 1.0, "sampler_name": "uni_pc", "scheduler": "simple",
                     "start_at_step": 5, "end_at_step": 10000, "return_with_leftover_noise": "disable"},
          "class_type": "KSamplerAdvanced"},
  "8":   {"inputs": {"samples": ["191", 0], "vae": ["39", 0]}, "class_type": "VAEDecode"},
  "135": {"inputs": {"images": ["8", 0], "frame_rate": 16, "loop_count": 0,
                     "filename_prefix": "wan22_i2v", "format": "video/h264-mp4",
                     "pingpong": False, "save_output": True},
          "class_type": "VHS_VideoCombine"}
}

workflows_to_write = {
    "ltx_t2v.json": ltx_t2v,
    "ltx_i2v.json": ltx_t2v,   # placeholder — override with real i2v template if available
    "wan_t2v.json": wan_t2v,
    "wan22_t2i.json": wan22_t2i,
    "wan22_i2v.json": wan22_i2v,
}
for fname, wf in workflows_to_write.items():
    dest = f"{BACKEND_DIR}/workflows/{fname}"
    if not os.path.exists(dest):
        with open(dest, "w") as f:
            json.dump(wf, f, indent=2)
        print(f"  ✓ Wrote {fname}")
    else:
        print(f"  ✓ {fname} already present (from Kaggle input)")

print("\n✅ Backend files ready.")

In [ ]:
# ============================================================
# CELL 8 — OPTIONAL: Import your associate's ComfyUI workflow
# ============================================================
# If your associate exported their ComfyUI workflow as
# "API format" JSON, you can import it here.
# 1. Export from ComfyUI: Settings → Save (API Format)
# 2. Upload the JSON as a Kaggle dataset
# 3. Set the path below

ASSOCIATE_WORKFLOW_PATH = ""  # e.g. "/kaggle/input/my-workflow/workflow_api.json"
ASSOCIATE_WORKFLOW_NAME = "my_custom_workflow"  # name it anything

if ASSOCIATE_WORKFLOW_PATH and os.path.exists(ASSOCIATE_WORKFLOW_PATH):
    import shutil
    dest = f"{STUDIO_DIR}/backend/workflows/{ASSOCIATE_WORKFLOW_NAME}.json"
    shutil.copy(ASSOCIATE_WORKFLOW_PATH, dest)
    print(f"✅ Workflow imported as '{ASSOCIATE_WORKFLOW_NAME}'")
    print(f"   Add it to models_config.json with workflow_file: '{ASSOCIATE_WORKFLOW_NAME}.json'")
else:
    print("ℹ️  No custom workflow to import (set ASSOCIATE_WORKFLOW_PATH to import one)")

In [ ]:
# ============================================================
# CELL 9 — Start ComfyUI
# ============================================================

import subprocess, time, requests

comfy_env = {
    **os.environ,
    "PYTHONPATH": COMFY_DIR,
}

comfy_proc = subprocess.Popen(
    [
        "python", f"{COMFY_DIR}/main.py",
        "--listen", "0.0.0.0",
        "--port", str(COMFY_PORT),
        "--output-directory", f"{COMFY_DIR}/output",
        "--input-directory", f"{COMFY_DIR}/input",
        "--disable-auto-launch",
    ],
    cwd=COMFY_DIR,
    env=comfy_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("Starting ComfyUI…", end="")
for _ in range(60):
    try:
        r = requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2)
        if r.status_code == 200:
            print(" ready!")
            break
    except:
        pass
    print(".", end="", flush=True)
    time.sleep(3)
else:
    print("\n⚠️  ComfyUI may not have started. Check logs.")

print(f"\nComfyUI running on http://127.0.0.1:{COMFY_PORT}")

In [ ]:
# ============================================================
# CELL 10 — Start AI Studio Backend
# ============================================================

import subprocess, os

studio_env = {
    **os.environ,
    "COMFY_HOST": "127.0.0.1",
    "COMFY_PORT": str(COMFY_PORT),
    "COMFY_DIR": COMFY_DIR,
    "COMFY_OUTPUT_DIR": f"{COMFY_DIR}/output",
}

# Pass FireRed model paths if downloaded
if DOWNLOAD_FIRERED:
    studio_env["FIRERED_MODEL_ID"] = f"{WORK_DIR}/hf_cache/FireRed-Image-Edit-1.1"
    studio_env["FIRERED_TRANSFORMER_ID"] = f"{WORK_DIR}/hf_cache/Qwen-Image-Edit-Rapid-AIO-V19"
    studio_env["HUGGINGFACE_HUB_CACHE"] = f"{WORK_DIR}/hf_cache"

if HF_TOKEN:
    studio_env["HF_TOKEN"] = HF_TOKEN
    studio_env["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

studio_proc = subprocess.Popen(
    [
        "python", "-m", "uvicorn", "server:app",
        "--host", "0.0.0.0",
        "--port", str(STUDIO_PORT),
    ],
    cwd=f"{STUDIO_DIR}/backend",
    env=studio_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

import time, requests
print("Starting AI Studio backend…", end="")
for _ in range(20):
    try:
        r = requests.get(f"http://127.0.0.1:{STUDIO_PORT}/api/status", timeout=2)
        if r.status_code == 200:
            print(" ready!")
            break
    except:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print("\n⚠️  Studio backend may not be ready.")

print(f"\nAI Studio backend on http://127.0.0.1:{STUDIO_PORT}")

In [ ]:
# ============================================================
# CELL 11 — Create Public Tunnel (cloudflared)
# ============================================================
# This creates a public HTTPS URL that you can open in your
# browser from anywhere.

import subprocess, time, re, threading

# Install cloudflared if needed
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Installing cloudflared…")
    run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
    run("chmod +x /usr/local/bin/cloudflared")

tunnel_url = None
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{STUDIO_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True
)

print("Creating tunnel", end="")
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    print(".", end="", flush=True)
    time.sleep(1)

print()
if tunnel_url:
    print("\n" + "="*60)
    print("🎬  AI STUDIO IS READY")
    print("="*60)
    print(f"\n  👉  {tunnel_url}")
    print("\n" + "="*60)
    print("Open the URL above in your browser.")
    print("Bookmark it — it changes each session.")
else:
    print("⚠️  Could not get tunnel URL. Try ngrok instead (see next cell).")
    print(f"   Local access: http://localhost:{STUDIO_PORT}")

In [ ]:
# ============================================================
# CELL 12 — Alternative: ngrok tunnel
# ============================================================
# Use this if cloudflared doesn't work.
# You need a free ngrok account: https://ngrok.com

NGROK_TOKEN = ""  # paste your ngrok authtoken here

if NGROK_TOKEN:
    run("pip install -q pyngrok")
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    public_url = ngrok.connect(STUDIO_PORT)
    print("\n" + "="*60)
    print("🎬  AI STUDIO IS READY (ngrok)")
    print("="*60)
    print(f"\n  👉  {public_url}")
    print("\n" + "="*60)
else:
    print("ℹ️  Set NGROK_TOKEN if you want to use ngrok instead of cloudflared")

In [ ]:
# ============================================================
# CELL 13 — Keep alive (run this to prevent Kaggle timeout)
# ============================================================
# Kaggle sessions expire after ~9h idle.
# This cell keeps the process alive and shows live logs.

import time, sys

print("Session alive. ComfyUI + AI Studio are running.")
print("Interrupt this cell (■) to stop them.")
print(f"Studio URL: {tunnel_url}" if tunnel_url else f"Local: http://localhost:{STUDIO_PORT}")
print()

try:
    while True:
        # Optionally print ComfyUI log lines for debugging:
        # line = comfy_proc.stdout.readline()
        # if line: print("[comfy]", line.rstrip())
        time.sleep(10)
except KeyboardInterrupt:
    print("\nStopping processes…")
    comfy_proc.terminate()
    studio_proc.terminate()
    print("Stopped.")